# Scaling Numerical Features Using StandardScaler

Many machine learning models are sensitive to feature scale. When numerical inputs live on very different ranges, optimization, distance calculations, and coefficient estimates can become biased toward large-magnitude variables.

This lesson explains why scaling matters, when to apply it, how StandardScaler works, and how to avoid data leakage while scaling.

## Why scaling is necessary

Models such as logistic regression, linear regression with regularization, support vector machines, k-nearest neighbors, neural networks, and PCA often benefit from standardized inputs.

Scaling helps because:

- Optimization becomes more stable.
- Distance-based methods treat features more fairly.
- Coefficients are less dominated by large numeric ranges.
- Numerical features become comparable across columns.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

rng = np.random.default_rng(19)
n_rows = 500

df = pd.DataFrame({
    'tenure': rng.integers(1, 72, size=n_rows),
    'MonthlyCharges': rng.normal(70, 15, size=n_rows).round(2),
    'TotalCharges': rng.normal(2500, 800, size=n_rows).round(2),
    'Contract': rng.choice(['Month-to-month', 'One year', 'Two year'], size=n_rows),
})

logit = -0.025 * df['tenure'] + 0.018 * df['MonthlyCharges'] + 0.0004 * df['TotalCharges']
prob = 1 / (1 + np.exp(-logit))
df['Churn'] = (rng.random(n_rows) < prob).astype(int)

df.head()

## How StandardScaler works

StandardScaler standardizes each numerical feature using:

scaled_value = (x - mean) / standard_deviation

After scaling, each feature has mean close to 0 and standard deviation close to 1. That makes features dimensionless and easier for many models to optimize.

In [ ]:
TARGET_COLUMN = 'Churn'
NUMERICAL_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']
CATEGORICAL_FEATURES = ['Contract']

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), NUMERICAL_FEATURES),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ]), CATEGORICAL_FEATURES),
])

model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print({'test_accuracy': round(accuracy, 3)})

## Avoiding leakage while scaling

Scaling must happen after the train-test split. The scaler should be fit on training data only, then reused to transform test and production data.

Incorrect pattern:

- Fit the scaler on the full dataset
- Split afterward

Correct pattern:

- Split first
- Fit the scaler on training data only
- Transform test data using the fitted scaler

When you save the trained pipeline, the fitted scaler should be saved with it so inference never refits preprocessing.

## Practical checklist

- Scale only numerical features.
- Do not scale categorical labels or target values.
- Split before fitting any scaler.
- Use pipelines or ColumnTransformer for clean preprocessing.
- Save the fitted scaler with the model artifact.
- Verify that the transformed training data has mean near 0 and standard deviation near 1.

## Bonus resources

- Scikit-learn StandardScaler Documentation
- Pipeline and ColumnTransformer Guide